In [ ]:
pip install mysql-connector-python

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import seaborn as sns

orders = pd.read_csv(r'C:\Users\DELL\Downloads\orders.csv')
returns = pd.read_csv(r'C:\Users\DELL\Downloads\returns.csv')
products = pd.read_csv(r'C:\Users\DELL\Downloads\product_master.csv')
customers = pd.read_csv(r'C:\Users\DELL\Downloads\customer_master.csv')
stores = pd.read_csv(r'C:\Users\DELL\Downloads\store_master.csv')


In [ ]:
print(orders.head())

In [ ]:
print(orders.describe())

In [ ]:
print(orders.info())

#### 2. Check data types, missing values, duplicates, unique values:

In [ ]:
orders.isnull().sum()

In [ ]:
orders.duplicated().sum()

#### Merge all datasets to get a one dataset (sales):

In [ ]:
# Merge Orders with Product, Customer, Store info
df = orders.merge(products, on='ProductID', how='left') \
           .merge(customers, on='CustomerID', how='left') \
           .merge(stores, on='StoreID', how='left')


In [ ]:
# Merge returns if needed for return analysis
df = df.merge(returns[['OrderID', 'Reason']], on='OrderID', how='left')

In [ ]:
print(df.columns)

#### Step 3: Data Cleaning
Remove duplicates:

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df['Reason'] = df['Reason'].fillna('No Return')

In [ ]:
df['Margin'] = df['Revenue'] - df['GMV'] 
df['ReturnFlag'] = df['Reason'].apply(lambda x: 0 if x == 'No Return' else 1)

#### Step 4: KPI

In [ ]:
total_sales = df['Revenue'].sum()
total_gmv = df['GMV'].sum()
total_margin = df['Margin'].sum()
avg_aov = df['Revenue'].sum() / df['OrderID'].nunique()
total_units = df['Quantity'].sum()
return_rate = df['ReturnFlag'].mean() * 100
repeat_rate = (df.groupby('CustomerID')['OrderID'].nunique() > 1).mean() * 100

In [ ]:
total_sales

In [ ]:
total_gmv

In [ ]:
total_margin

In [ ]:
avg_aov

In [ ]:
total_units

In [ ]:
return_rate

In [ ]:
repeat_rate

In [ ]:
kpi_names = ['Total Sales', 'GMV', 'Total Margin', 'AOV', 'Total Units', 'Return Rate', 'Repeat Rate']
kpi_values = [total_sales, total_gmv, total_margin, avg_aov, total_units, return_rate, repeat_rate]

plt.figure(figsize=(12,6))
sns.barplot(x=kpi_names, y=kpi_values, palette='viridis')
plt.title("Sales Analysis360 KPI")
plt.ylabel("Value")
plt.xticks(rotation=45)
plt.show()

### Top 10 Revenue

In [ ]:
top_products = df.groupby('ProductName')['Revenue'].sum().sort_values(ascending=False).head(10)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.barplot(x=top_products.values, y=top_products.index)
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.show()

#### Bottom 10 Revenue

In [ ]:
# Bottom 10 Products by Revenue
bottom_products = df.groupby('ProductName')['Revenue'].sum().sort_values(ascending=True).head(10)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.barplot(x=bottom_products.values, y=bottom_products.index)
plt.title("Bottom 10 Products by Revenue")
plt.xlabel("Revenue")
plt.show()

In [ ]:
# Category-wise Revenue
category_sales = df.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=category_sales.values, y=category_sales.index)
plt.title("Revenue by Category")
plt.show()

In [ ]:
# Brand-wise Revenue
brand_sales = df.groupby('Brand')['Revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,5))
sns.barplot(x=brand_sales.values, y=brand_sales.index)
plt.title("Revenue by Brand")
plt.show()

#### Store & Region Analysis

In [ ]:
store_sales = df.groupby('StoreName')['Revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(8,6))
sns.barplot(x=store_sales.values, y=store_sales.index)
plt.title("Revenue by Store")
plt.show()

#### City-wise Revenue

In [ ]:
city_sales = df.groupby('City_y')['Revenue'].sum().sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=city_sales.values, y=city_sales.index)
plt.title("Revenue by City")
plt.show()

#### Return Analysis 

In [ ]:
df['OrderCount'] = df.groupby('CustomerID')['OrderID'].transform('count')
df['CustomerType'] = df['OrderCount'].apply(lambda x: 'New' if x==1 else 'Returning')
customer_type_sales = df.groupby('CustomerType')['Revenue'].sum()
plt.figure(figsize=(6,6))
customer_type_sales.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title("Revenue by Customer Type")
plt.ylabel("")
plt.show()


In [ ]:
# RFM Analysis (Frequency & Monetary only)
rfm = df.groupby('CustomerID').agg({
    'OrderID': 'count',
    'Revenue': 'sum'
}).rename(columns={'OrderID':'Frequency','Revenue':'Monetary'})
print("RFM Sample:")
print(rfm.head())


In [ ]:
# Return Reasons
return_reason = df['Reason'].value_counts()
plt.figure(figsize=(8,5))
sns.barplot(x=return_reason.values, y=return_reason.index)
plt.title("Return Reasons")
plt.xlabel("Count")
plt.show()

In [ ]:
# Top 10 Products by Return Rate
return_by_product = df.groupby('ProductName')['ReturnFlag'].mean().sort_values(ascending=False).head(10)
plt.figure(figsize=(10,6))
sns.barplot(x=return_by_product.values, y=return_by_product.index)
plt.title("Top 10 Products by Return Rate")
plt.xlabel("Return Rate")
plt.show()

In [ ]:
# Correlation Analysis 
numeric_cols = ['Revenue','GMV','Margin','Quantity']
plt.figure(figsize=(6,5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

#### EDA Analysis Pattern

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# AOV Distribution
# Calculate AOV per order
order_aov = df.groupby('OrderID')['Revenue'].sum()

plt.figure(figsize=(10,5))
sns.histplot(order_aov, bins=30, kde=True, color='skyblue')
plt.title("Distribution of Average Order Value (AOV)")
plt.xlabel("Order Revenue")
plt.ylabel("Number of Orders")
plt.show()

In [ ]:
# 2. Region Revenue Heatmap (Bar Chart)
region_sales = df.groupby('City_y')['Revenue'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=region_sales.values, y=region_sales.index, palette='coolwarm')
plt.title("Revenue by Region / City")
plt.xlabel("Revenue")
plt.ylabel("City")
plt.show()

In [ ]:
# 3. Anomaly Detection (Extreme Outliers)
# Define anomalies as revenue > mean + 3*std
revenue_mean = df['Revenue'].mean()
revenue_std = df['Revenue'].std()
threshold = revenue_mean + 3*revenue_std

anomalies = df[df['Revenue'] > threshold]

print(f"Number of revenue anomalies: {len(anomalies)}")
print("Top 5 Revenue Anomalies:")
print(anomalies[['OrderID','ProductName','Revenue','Quantity']].sort_values(by='Revenue', ascending=False).head())



In [ ]:
#visualize anomalies
plt.figure(figsize=(12,6))
sns.scatterplot(x=range(len(df)), y=df['Revenue'], label='Revenue')
sns.scatterplot(x=anomalies.index, y=anomalies['Revenue'], color='red', label='Anomalies')
plt.title("Revenue Anomaly Detection")
plt.xlabel("Order Index")
plt.ylabel("Revenue")
plt.legend()
plt.show()

In [ ]:
.....................

In [ ]:
..............

......................